# NHANES liver FibroScan (LSM + CAP) distributions - overview

Estimates the U.S.-adult population distributions of the two NHANES transient-
elastography measurements, per (sex, 5-year age band), for the consuming
microsimulation (`vivarium_csu_mace_rct`):

- **LSM** - liver stiffness (kPa, `LUXSMED`) -> **fibrosis stage** (F0-F4)
- **CAP** - controlled attenuation parameter (dB/m, `LUXCAPM`) -> hepatic **steatosis**

## What the simulation needs

The simulation routes each simulant to an excess-mortality rate by **fibrosis
stage**, so the load-bearing quantity is the population **stage-share vector**.
The project priority is accuracy at **F1/F2/F3** -- the stages carrying most of the
population -- ahead of F4 (cirrhosis). Stage cutoffs are the repo-standard
**6 / 8 / 10 / 15 kPa** (F0<6, F1 6-8, F2 8-10, F3 10-15, F4>=15), matching
`nhanes_fibrosis_modeling`.

## Data source: two pooled NHANES cycles (2017-2023)

FibroScan first appears in the **2017 - March 2020 pre-pandemic release
(`P_LUX`, weight `WTMECPRP`)** and continues in **2021 - August 2023 (`LUX_L`,
weight `WTMEC2YR`)**. We **pool both cycles** (each MEC weight halved), which
roughly doubles the analytic sample and tightens every per-cell estimate. Age is
**top-coded at 80** in both cycles, so the terminal band is an open-ended 80+ cell.

## Method (notebook 04)

LSM is fit as a two-parameter **lognormal** -- the downstream sampler's contract --
whose `(mean_kpa, sd_kpa)` are chosen to **minimise the weighted stage-share error,
prioritising F1/F2/F3** (with small-cell targets smoothed across age). CAP is a
moment-matched **Normal** per cell. A single lognormal cannot also reproduce the
heavy >=15 kPa tail, so F4 is deliberately traded for F1/F2/F3 accuracy
(quantified in notebook 06).

## Notebooks

| # | Notebook | Purpose |
| --- | --- | --- |
| 01 | `01_download_lux.ipynb` | Download + pool both cycles; carry LSM and CAP; write pooled parquet |
| 02 | `02_lsm_marginal.ipynb` | Weighted LSM + CAP age x sex marginals; fibrosis stage-share profiles |
| 03 | `03_lsm_transformations.ipynb` | Transformation/outlier robustness; stage-share goodness of fit |
| 04 | `04_lsm_age_sex_calibration.ipynb` | **Core fit**: multi-cutoff lognormal (LSM) + moment-matched Normal (CAP); writes outputs |
| 05 | `05_extend_to_skeleton.ipynb` | Forward-fill both tables over the GBD demographic skeleton |
| 06 | `06_method_comparison.ipynb` | Methods side by side on stage-share accuracy; why multi-cutoff |
| 07 | `07_categorical_comparison.ipynb` | Investigation: a two-level (categorical joint stage + within-stage continuous) alternative vs the continuous fit |

## Outputs (`outputs/`)

- `liver_stiffness_age_sex_lognormal.csv` - LSM loader table (`mean_kpa`, `sd_kpa`
  + empirical/fitted stage shares + provenance)
- `cap_age_sex_distribution.csv` - CAP `(cap_mean, cap_sd)` + steatosis-grade shares
- `lsm_cap_calibration.meta.json` - cutoff ladders, stage weights, calibration objective


## Headline results (loaded from the committed outputs)

In [1]:
import json
from pathlib import Path
import pandas as pd

OUT = Path('outputs')
meta = json.load(open(OUT / 'lsm_cap_calibration.meta.json'))
print('LSM cutoffs (kPa):', meta['lsm_cutoffs_kpa'], '| stage weights:', meta['lsm_stage_weights'])
print('CAP family:', meta['cap_dist_family'], '| pooled LSM 60+ n =', meta.get('n_pooled_lsm_60plus'))
print()

lsm = pd.read_csv(OUT / 'liver_stiffness_age_sex_lognormal.csv')
fitted = lsm[lsm['source'] == 'fitted']
print('LSM fitted cells (mean_kpa, sd_kpa) with F1/F2/F3 empirical vs fitted shares:')
cols = ['sex', 'age_group', 'mean_kpa', 'sd_kpa',
        'lsm_f1_share', 'lsm_f1_fit', 'lsm_f2_share', 'lsm_f2_fit', 'lsm_f3_share', 'lsm_f3_fit']
print(fitted[cols].round(3).to_string(index=False))


LSM cutoffs (kPa): [6.0, 8.0, 10.0, 15.0] | stage weights: [1.0, 2.0, 2.0, 2.0, 0.5]
CAP family: normal | pooled LSM 60+ n = 5005

LSM fitted cells (mean_kpa, sd_kpa) with F1/F2/F3 empirical vs fitted shares:
   sex age_group  mean_kpa  sd_kpa  lsm_f1_share  lsm_f1_fit  lsm_f2_share  lsm_f2_fit  lsm_f3_share  lsm_f3_fit
Female     60-64     5.022   2.516         0.164       0.159         0.043       0.066         0.045       0.040
Female     65-69     5.110   2.508         0.165       0.166         0.052       0.069         0.052       0.041
Female     70-74     5.159   2.335         0.175       0.177         0.058       0.069         0.039       0.037
Female     75-79     5.180   2.064         0.188       0.190         0.040       0.064         0.022       0.027
Female       80+     5.192   1.765         0.198       0.203         0.045       0.055         0.021       0.016
  Male     60-64     5.622   2.416         0.215       0.214         0.063       0.090         0.053       0.049


In [2]:
cap = pd.read_csv(OUT / 'cap_age_sex_distribution.csv')
capf = cap[cap['source'] == 'fitted']
print('CAP fitted cells (dB/m):')
print(capf[['sex', 'age_group', 'cap_mean', 'cap_sd']].round(1).to_string(index=False))
print()
print('See notebook 06 for the method comparison: multi-cutoff calibration cuts the')
print('mean F1/F2/F3 stage-share error to ~1.3 pp (vs ~3.5-3.9 pp for the old')
print('F4-calibrated and log-moment-match methods), trading F4-tail accuracy.')


CAP fitted cells (dB/m):
   sex age_group  cap_mean  cap_sd
Female     60-64     264.6    58.2
Female     65-69     274.9    57.3
Female     70-74     263.2    58.5
Female     75-79     258.8    50.8
Female       80+     254.1    55.3
  Male     60-64     279.7    61.8
  Male     65-69     275.3    59.5
  Male     70-74     273.2    66.1
  Male     75-79     280.9    58.8
  Male       80+     261.6    61.0

See notebook 06 for the method comparison: multi-cutoff calibration cuts the
mean F1/F2/F3 stage-share error to ~1.3 pp (vs ~3.5-3.9 pp for the old
F4-calibrated and log-moment-match methods), trading F4-tail accuracy.
